Ignore 'Preprocess and Join Weather Data' Section, instead run following code

In [2]:
import pandas as pd
import sqlite3
from scipy.spatial import cKDTree
import numpy as np

df = pd.read_csv('../data/processed/data.csv')
df.head()

/var/folders/z_/h64hdrrd3vvdmcr314s1399w0000gn/T/ipykernel_45468/1167316539.py:6: DtypeWarning: Columns (17,19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/processed/data.csv')


,SOURCE_SYSTEM_TYPE,NWCG_REPORTING_AGENCY,FIRE_YEAR,DISCOVERY_DATE,DISCOVERY_DOY,DISCOVERY_TIME,STAT_CAUSE_CODE,STAT_CAUSE_DESCR,CONT_DATE,CONT_DOY,...,FIPS_NAME,weather_lat,weather_lng,lat,lng,elevation_m,date,prcp_mm/day,tmax_deg_c,vpd_Pa
0,FED,FS,2005,2005-02-02,33,1300.0,9.0,Miscellaneous,2453403.5,33.0,...,Plumas,39.4874,-120.7103,39.4874,-120.7103,1392.0,2005-02-02,0.0,11.96,821.232978
1,FED,FS,2004,2004-05-12,133,845.0,1.0,Lightning,2453137.5,133.0,...,Placer,38.3643,-120.4212,38.3643,-120.4212,1020.0,2004-05-12,0.0,22.19,1878.987757
2,FED,FS,2004,2004-05-31,152,1921.0,5.0,Debris Burning,2453156.5,152.0,...,El Dorado,39.4874,-120.7103,39.4874,-120.7103,1392.0,2004-05-31,0.0,26.17,2638.217170
3,FED,FS,2004,2004-06-28,180,1600.0,1.0,Lightning,2453189.5,185.0,...,Alpine,38.3643,-120.4212,38.3643,-120.4212,1020.0,2004-06-28,0.0,31.01,3898.723372
4,FED,FS,2004,2004-06-28,180,1600.0,1.0,Lightning,2453189.5,185.0,...,Alpine,38.3643,-120.4212,38.3643,-120.4212,1020.0,2004-06-28,0.0,31.01,3898.723372


### Preprocess and Join Weather Data

In [2]:
import pandas as pd
import sqlite3

weather = pd.read_csv('../data/raw/weather_1992_2015_combined.csv')

conn = sqlite3.connect('../data/raw/FPA_FOD_20170508.sqlite')
fire = pd.read_sql_query("SELECT * FROM Fires", conn)
conn.close()

In [3]:
fire.drop(columns=['FOD_ID', 'FPA_ID', 'LOCAL_FIRE_REPORT_ID', 'LOCAL_INCIDENT_ID', 
             'FIRE_CODE', 'FIRE_NAME', 'ICS_209_INCIDENT_NUMBER', 'ICS_209_NAME', 
             'MTBS_ID', 'MTBS_FIRE_NAME', 'COMPLEX_NAME', 'SOURCE_REPORTING_UNIT', 
             'SOURCE_REPORTING_UNIT_NAME','OBJECTID', 'SOURCE_SYSTEM', 'NWCG_REPORTING_UNIT_ID',
            'NWCG_REPORTING_UNIT_NAME', 'OWNER_CODE', 'Shape'] , inplace=True)

In [4]:
fire['DISCOVERY_DATE'] = pd.to_datetime(fire['FIRE_YEAR'].astype(str), format='%Y') + \
                               pd.to_timedelta(fire['DISCOVERY_DOY'] - 1, unit='D')

In [5]:
weather['date'] = pd.to_datetime(weather['date'].astype(str), format='%Y%m%d')

In [6]:
from scipy.spatial import cKDTree
import numpy as np

# Deduplicate weather locations
# Remove non-finite lat/lng values (NaN, inf, -inf)
weather_locs = weather[['lat', 'lng']].replace([np.inf, -np.inf], np.nan).dropna().drop_duplicates().reset_index(drop=True)

# Build KDTree from weather locations
weather_tree = cKDTree(weather_locs[['lat', 'lng']].to_numpy())

# Query closest weather location for each fire
fire_coords = fire[['LATITUDE', 'LONGITUDE']].to_numpy()
_, indices = weather_tree.query(fire_coords, k=1)

# Attach nearest lat/lng to wildfire DataFrame
nearest_coords = weather_locs.iloc[indices].reset_index(drop=True)
fire['weather_lat'] = nearest_coords['lat'].round(4)
fire['weather_lng'] = nearest_coords['lng'].round(4)
weather['lat'] = weather['lat'].round(4)
weather['lng'] = weather['lng'].round(4)
# Merge wildfire with weather on lat/lng/date
df = pd.merge(
    fire,
    weather,
    how='left',
    left_on=['weather_lat', 'weather_lng', 'DISCOVERY_DATE'],
    right_on=['lat', 'lng', 'date'],
    suffixes=('', '_weather')
)


In [7]:
df[['FIRE_YEAR', 'DISCOVERY_DATE', 'LATITUDE', 'LONGITUDE', 'weather_lat', 'weather_lng', 'tavg']].head()

,FIRE_YEAR,DISCOVERY_DATE,LATITUDE,LONGITUDE,weather_lat,weather_lng,tavg
0,2005,2005-02-02,40.036944,-121.005833,40.0046,-120.8385,3.812
1,2004,2004-05-12,38.933056,-120.404444,38.7787,-120.5247,8.179
2,2004,2004-05-31,38.984167,-120.735556,39.0634,-120.7177,16.600
3,2004,2004-06-28,38.559167,-119.913333,38.5972,-119.8207,14.430
4,2004,2004-06-28,38.559167,-119.933056,38.5972,-119.8207,14.430


### Model Dev

In [13]:
#STAT_CAUSE_DESCR, FIRE_SIZE, FIRE_SIZE_CLASS, STATE, tavg

In [3]:
# Split by year
train_df = df[df["FIRE_YEAR"] < 2015]
test_df  = df[df["FIRE_YEAR"] == 2015]

X_train = train_df[['STAT_CAUSE_DESCR', 'FIRE_SIZE', 'FIRE_SIZE_CLASS', 'STATE', 'elevation_m', 'prcp_mm/day', 'tmax_deg_c', 'vpd_Pa']]
y_train = train_df[["LATITUDE", "LONGITUDE"]]

X_test  = test_df[['STAT_CAUSE_DESCR', 'FIRE_SIZE', 'FIRE_SIZE_CLASS', 'STATE', 'elevation_m', 'prcp_mm/day', 'tmax_deg_c', 'vpd_Pa']]
y_test  = test_df[["LATITUDE", "LONGITUDE"]]

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

X_train.head()

Train shape: (1805974, 29)
Test shape: (74491, 29)


,STAT_CAUSE_DESCR,FIRE_SIZE,FIRE_SIZE_CLASS,STATE,elevation_m,prcp_mm/day,tmax_deg_c,vpd_Pa
0,Miscellaneous,0.10,A,CA,1392.0,0.0,11.96,821.232978
1,Lightning,0.25,A,CA,1020.0,0.0,22.19,1878.987757
2,Debris Burning,0.10,A,CA,1392.0,0.0,26.17,2638.217170
3,Lightning,0.10,A,CA,1020.0,0.0,31.01,3898.723372
4,Lightning,0.10,A,CA,1020.0,0.0,31.01,3898.723372


In [4]:
X_train = X_train.fillna(X_train.median(numeric_only=True))
X_test  = X_test.fillna(X_train.median(numeric_only=True))

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X_train.select_dtypes(include=["int64","float64"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols)
    ]
)

# Fit on training, transform both train/test
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

In [6]:
# Inputs: use your existing preprocessor (handles numeric + categorical)
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Outputs (lat, lon): scale separately
y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train)   # fit on pre-2015
y_test_scaled = y_scaler.transform(y_test)        # transform 2015

In [7]:
import torch

X_train_tensor = torch.tensor(X_train_processed, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test_processed, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)
y_test_tensor  = torch.tensor(y_test_scaled, dtype=torch.float32)

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim

# Neural Net to predict (Latitude, Longitude)
import torch.nn as nn

class FireLocationNN(nn.Module):
    def __init__(self, input_dim):
        super(FireLocationNN, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 2)   # 🔹 Output 2 values: [lat, lon]

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x


### Training Model Pre 2015

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim

# 🔥 Improved NN for Latitude + Longitude prediction
class FireLocationNN(nn.Module):
    def __init__(self, input_dim):
        super(FireLocationNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),   # bigger first layer
            nn.BatchNorm1d(256),         # normalize activations
            nn.ReLU(),
            nn.Dropout(0.3),             # reduce overfitting

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 2)             # output = [lat, lon]
        )

    def forward(self, x):
        return self.net(x)


# --- Training setup ---
model = FireLocationNN(X_train_tensor.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 50
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [1/50], Loss: 1.0430
Epoch [2/50], Loss: 0.9437
Epoch [3/50], Loss: 0.8669
Epoch [4/50], Loss: 0.8045
Epoch [5/50], Loss: 0.7524
Epoch [6/50], Loss: 0.7098
Epoch [7/50], Loss: 0.6725
Epoch [8/50], Loss: 0.6388
Epoch [9/50], Loss: 0.6065
Epoch [10/50], Loss: 0.5724
Epoch [11/50], Loss: 0.5387
Epoch [12/50], Loss: 0.5047
Epoch [13/50], Loss: 0.4717
Epoch [14/50], Loss: 0.4410
Epoch [15/50], Loss: 0.4112
Epoch [16/50], Loss: 0.3842
Epoch [17/50], Loss: 0.3590
Epoch [18/50], Loss: 0.3364
Epoch [19/50], Loss: 0.3174
Epoch [20/50], Loss: 0.3017
Epoch [21/50], Loss: 0.2890
Epoch [22/50], Loss: 0.2784
Epoch [23/50], Loss: 0.2687
Epoch [24/50], Loss: 0.2595
Epoch [25/50], Loss: 0.2498
Epoch [26/50], Loss: 0.2395
Epoch [27/50], Loss: 0.2291
Epoch [28/50], Loss: 0.2185
Epoch [29/50], Loss: 0.2088
Epoch [30/50], Loss: 0.1992
Epoch [31/50], Loss: 0.1908
Epoch [32/50], Loss: 0.1830
Epoch [33/50], Loss: 0.1761
Epoch [34/50], Loss: 0.1698
Epoch [35/50], Loss: 0.1643
Epoch [36/50], Loss: 0.1592
E

### Model on 2015 Data

In [11]:
import torch
import joblib   # for saving scalers + numpy objects
import numpy as np

# --- EVALUATE FULL TEST SET ---
model.eval()
with torch.no_grad():
    y_pred_scaled = model(X_test_tensor).cpu().numpy()
    y_pred = y_scaler.inverse_transform(y_pred_scaled)   # back to real lat/lon
    y_true = y_scaler.inverse_transform(y_test_tensor.cpu().numpy())

# Save predictions to .npy or .csv
np.save("fire_predictions.npy", y_pred)
np.save("fire_true.npy", y_true)

# Optional: save to CSV if you prefer
import pandas as pd
df_preds = pd.DataFrame(y_pred, columns=["Pred_Lat", "Pred_Lon"])
df_true  = pd.DataFrame(y_true, columns=["True_Lat", "True_Lon"])
df_out = pd.concat([df_preds, df_true], axis=1)
df_out.to_csv("fire_predictions.csv", index=False)

print("✅ Saved predictions to fire_predictions.npy and fire_predictions.csv")

# --- SAVE MODEL + SCALER ---
torch.save(model.state_dict(), "fire_model.pth")   # save weights
joblib.dump(y_scaler, "y_scaler.pkl")              # save scaler

print("✅ Saved model and scalers")

✅ Saved predictions to fire_predictions.npy and fire_predictions.csv
✅ Saved model and scalers


To implement model again without retraining it: 

```python
# Recreate model with same architecture
model = FireLocationNN(X_test_tensor.shape[1])
model.load_state_dict(torch.load("fire_model.pth"))
model.eval()

# Reload scalers
y_scaler = joblib.load("y_scaler.pkl")
X_scaler = joblib.load("X_scaler.pkl")

# Predict again
with torch.no_grad():
    y_pred_scaled = model(X_test_tensor).cpu().numpy()
    y_pred = y_scaler.inverse_transform(y_pred_scaled)

print(y_pred[:5])  # show first 5 predictions
```

### Evaluating Model 

In [12]:
import numpy as np

# Percent error for each coordinate
lat_error = np.abs((y_pred[:,0] - y_true[:,0]) / y_true[:,0]) * 100
lon_error = np.abs((y_pred[:,1] - y_true[:,1]) / y_true[:,1]) * 100

print("Mean Latitude Percent Error:", np.mean(lat_error))
print("Mean Longitude Percent Error:", np.mean(lon_error))

Mean Latitude Percent Error: 3.7956767
Mean Longitude Percent Error: 3.2202733


In [13]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

# Make predictions on 2015 test set
model.eval()
with torch.no_grad():
    y_pred_scaled = model(X_test_tensor).cpu().numpy()
    y_pred = y_scaler.inverse_transform(y_pred_scaled)   # back to real lat/lon
    y_true = y_scaler.inverse_transform(y_test_tensor.cpu().numpy())

# --- Metrics ---
mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_true, y_pred)

# Lat / Lon split
lat_mse = mean_squared_error(y_true[:,0], y_pred[:,0])
lon_mse = mean_squared_error(y_true[:,1], y_pred[:,1])
lat_rmse = np.sqrt(lat_mse)
lon_rmse = np.sqrt(lon_mse)

print(f"Overall MSE: {mse:.4f}, RMSE: {rmse:.4f}, MAE: {mae:.4f}")
print(f"Latitude RMSE: {lat_rmse:.4f}, Longitude RMSE: {lon_rmse:.4f}")

Overall MSE: 11.8452, RMSE: 3.4417, MAE: 2.3056
Latitude RMSE: 2.0597, Longitude RMSE: 4.4100


In [14]:
import haversine as haverine
from haversine import haversine
import numpy as np

# Convert lat/lon pairs into tuples
y_true_coords = [tuple(coord) for coord in y_true]
y_pred_coords = [tuple(coord) for coord in y_pred]

# Compute haversine distance (km) for each prediction
distances = [haversine(y_true_coords[i], y_pred_coords[i]) for i in range(len(y_true_coords))]

print("Mean distance error (km):", np.mean(distances))
print("Median distance error (km):", np.median(distances))
print("Max distance error (km):", np.max(distances))

Mean distance error (km): 337.35775307489865
Median distance error (km): 281.0073372056663
Max distance error (km): 4431.848336938182


### Visualize Model Predictions

In [15]:
print("Prediction shape:", y_pred.shape)
print("Sample predictions:", y_pred[:10])
print("Min/Max Latitude:", np.min(y_pred[:,0]), np.max(y_pred[:,0]))
print("Min/Max Longitude:", np.min(y_pred[:,1]), np.max(y_pred[:,1]))

Prediction shape: (74491, 2)
Sample predictions: [[  44.04494  -110.05772 ]
 [  45.02738  -112.63998 ]
 [  44.236237 -110.1644  ]
 [  43.64336  -109.70321 ]
 [  45.289013 -109.99421 ]
 [  44.507225 -110.32226 ]
 [  44.891647 -109.4289  ]
 [  44.621338 -110.65028 ]
 [  43.95035  -109.69432 ]
 [  43.81231  -110.2091  ]]
Min/Max Latitude: 22.102211 58.4037
Min/Max Longitude: -141.33856 -69.857834


In [16]:
import folium
from folium.plugins import HeatMap
import numpy as np

# Ensure model is in eval mode
model.eval()

with torch.no_grad():
    y_pred_scaled = model(X_test_tensor).numpy()
    y_pred = y_scaler.inverse_transform(y_pred_scaled)  # back to real lat/lon
    y_true = y_scaler.inverse_transform(y_test_tensor.numpy())

# Convert to list of [lat, lon]
pred_coords = [[lat, lon] for lat, lon in y_pred]

# Create a folium map centered on mean location
m = folium.Map(location=[np.mean(y_pred[:,0]), np.mean(y_pred[:,1])], zoom_start=5)

# Add heatmap with smaller, sharper blobs
HeatMap(pred_coords, radius=3, blur=2, max_zoom=8).add_to(m)
m